# 07 Giải thích pretrained và quy trình cải tiến C1 C2 C3

Notebook bổ sung cho bản nâng cấp 29/10/2026. Kết quả benchmark gốc giữ ở `results/final_submission`.
Các cấu hình mới là giả thuyết thử nghiệm, chưa có F1 full chứng minh cải thiện.
Chạy demo từ checkpoint đã lưu không cần chạy các cell huấn luyện.
Nguồn lý thuyết và phân tích XED ở Phụ lục C của báo cáo.

In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Mở notebook từ repo hoặc notebooks/"
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
EXPERIMENTS = ROOT / "artifacts/improvements_notebook"
RUN_SMOKE = False
RUN_FULL_PILOT = False
RUN_CONFIRM = False
print("Không tự chạy huấn luyện khi mở notebook.")

## 1 Tính tay có thể kiểm lại

Ví dụ hai nhãn: attention, logit, sigmoid, BCE, gradient và một bước SGD.
Có kiểm tra gradient bằng sai phân hữu hạn. Dữ liệu tự đặt; optimizer thật là AdamW.

In [ ]:
from examples.pretrained_math import calculate
calculate()

## 2 Tạo kế hoạch trước khi train

Mỗi C có LR 2e-5/3e-5/5e-5, trần tám epoch, patience hai, min_delta 0,001.
Chọn bằng validation @0,5. Trần epoch làm thay đổi cả lịch LR tuyến tính.
Seed không có tính chất làm F1 cao hơn; giữ ba seed để đo dao động.

In [ ]:
command = [sys.executable, "-m", "scripts.transformers.plan_improvements",
           "--output-root", str(EXPERIMENTS / "pilot"), "--device", "cuda"]
subprocess.run(command, cwd=ROOT, check=True)
plan = json.loads((EXPERIMENTS / "pilot/plan.json").read_text(encoding="utf-8"))
[(r["architecture"], r["learning_rate"], r["seed"], r["status"]) for r in plan["runs"]]

## 3 C1 BERT cased

Đọc các bước: seed, tokenizer đúng checkpoint, train/validation riêng, head 28 logits,
BCE, backward, gradient clipping, AdamW, validation rồi lưu best checkpoint.
Cell dưới chỉ chạy smoke khi bật RUN_SMOKE. Smoke 64/32 không tạo benchmark.

In [ ]:
command = [sys.executable, "-m", "scripts.transformers.train_transformer",
           "--architecture", "bert", "--seed", "42", "--device", "cuda",
           "--smoke", "--diagnostics", "--patience", "2", "--min-delta", "0.001",
           "--output-root", str(EXPERIMENTS / "smoke"), "--resume"]
print(command)
if RUN_SMOKE: subprocess.run(command, cwd=ROOT, check=True)

## 4 C2 RoBERTa

Đọc các bước: seed, tokenizer đúng checkpoint, train/validation riêng, head 28 logits,
BCE, backward, gradient clipping, AdamW, validation rồi lưu best checkpoint.
Cell dưới chỉ chạy smoke khi bật RUN_SMOKE. Smoke 64/32 không tạo benchmark.

In [ ]:
command = [sys.executable, "-m", "scripts.transformers.train_transformer",
           "--architecture", "roberta", "--seed", "42", "--device", "cuda",
           "--smoke", "--diagnostics", "--patience", "2", "--min-delta", "0.001",
           "--output-root", str(EXPERIMENTS / "smoke"), "--resume"]
print(command)
if RUN_SMOKE: subprocess.run(command, cwd=ROOT, check=True)

## 5 C3 DistilBERT uncased

Đọc các bước: seed, tokenizer đúng checkpoint, train/validation riêng, head 28 logits,
BCE, backward, gradient clipping, AdamW, validation rồi lưu best checkpoint.
Cell dưới chỉ chạy smoke khi bật RUN_SMOKE. Smoke 64/32 không tạo benchmark.

In [ ]:
command = [sys.executable, "-m", "scripts.transformers.train_transformer",
           "--architecture", "distilbert", "--seed", "42", "--device", "cuda",
           "--smoke", "--diagnostics", "--patience", "2", "--min-delta", "0.001",
           "--output-root", str(EXPERIMENTS / "smoke"), "--resume"]
print(command)
if RUN_SMOKE: subprocess.run(command, cwd=ROOT, check=True)

## 6 Full pilot và xác nhận đủ ba seed

Bật RUN_FULL_PILOT khi có ngân sách. Sau pilot đầy đủ, confirm chọn một LR/kiến trúc
bằng validation và khóa revision, rồi chạy 42/123/2026 ở root khác.
Test hiện tại đã xem nên chỉ so sánh mô tả. Cần holdout mới cho đánh giá độc lập.

In [ ]:
if RUN_FULL_PILOT:
    subprocess.run([sys.executable, "-m", "scripts.transformers.plan_improvements",
                    "--output-root", str(EXPERIMENTS / "pilot"), "--device", "cuda", "--execute"], cwd=ROOT, check=True)
if RUN_CONFIRM:
    subprocess.run([sys.executable, "-m", "scripts.transformers.plan_improvements",
                    "--stage", "confirm", "--pilot-root", str(EXPERIMENTS / "pilot"),
                    "--output-root", str(EXPERIMENTS / "confirm"), "--device", "cuda", "--execute"], cwd=ROOT, check=True)

## 7 Chạy demo mà không train lại

Khôi phục checkpoint bộ nộp theo README nếu máy chưa có. Tạo gói mới bằng lệnh dưới,
sau đó giải nén GoEmotions_Web.zip và chạy START_DEMO.cmd.
Gói đã phát hành trong thư mục bản nâng cấp có sẵn model nên không cần dựng lại.
CSV của demo lưu điểm, không thay thế file mô hình.

In [ ]:
command = [sys.executable, "-m", "scripts.verification.package_demo",
           "--output", str(ROOT / "artifacts/portable_release_notebook")]
print("Chỉ chạy lệnh sau nếu cần tạo gói mới và thư mục đích chưa tồn tại:")
print(command)